In [1]:
import os
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from sklearn.preprocessing import LabelEncoder
import gc

DATA_PATH = '/kaggle/input/competitions/enveda-CASMI26-molecule-id-mass-spectra'
ID_COL = 'spectrum_id'
TARGET_COL = 'normalized_smiles'

# Get exact scalar columns
train_schema = pq.read_schema(os.path.join(DATA_PATH, 'train.parquet')).names
test_schema = pq.read_schema(os.path.join(DATA_PATH, 'test.parquet')).names

nested_cols = {'ms2_mzs', 'ms2_normalized_intensities', 'collision_energy_ev'}
common_features = [
    col for col in train_schema 
    if col in test_schema and col not in nested_cols and not col.startswith('__')
]

print("Loading metadata...")
train_df = pd.read_parquet(
    os.path.join(DATA_PATH, 'train.parquet'), 
    columns=common_features + [TARGET_COL]
)
test_df = pd.read_parquet(
    os.path.join(DATA_PATH, 'test.parquet'), 
    columns=common_features
)

# Downsample training data to 50,000 rows to guarantee low memory usage
if len(train_df) > 50000:
    train_df = train_df.sample(n=50000, random_state=42).reset_index(drop=True)

test_ids = test_df[ID_COL].copy() if ID_COL in test_df.columns else None
model_features = [col for col in common_features if col != ID_COL]

# Handle numerical columns
num_cols = train_df[model_features].select_dtypes(include=['int64', 'float64', 'float32']).columns.tolist()
for col in num_cols:
    med = train_df[col].median()
    train_df[col] = train_df[col].fillna(med).astype(np.float32)
    test_df[col] = test_df[col].fillna(med).astype(np.float32)

# Handle categorical columns
cat_cols = train_df[model_features].select_dtypes(include=['object', 'category', 'string']).columns.tolist()
for col in cat_cols:
    le = LabelEncoder()
    le.fit(pd.concat([train_df[col].astype(str), test_df[col].astype(str)]))
    train_df[col] = le.transform(train_df[col].astype(str)).astype(np.int32)
    test_df[col] = le.transform(test_df[col].astype(str)).astype(np.int32)

X = train_df[model_features].values.astype(np.float32)
X_test = test_df[model_features].values.astype(np.float32)

target_encoder = LabelEncoder()
y = target_encoder.fit_transform(train_df[TARGET_COL].astype(str))

del train_df
gc.collect()

print("Data ready!")

Loading metadata...
Data ready!


In [2]:
from sklearn.ensemble import RandomForestClassifier

print("Training model...")

# Fast, low-memory classifier
model = RandomForestClassifier(
    n_estimators=20,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)

model.fit(X, y)

print("Generating predictions...")
test_pred_indices = model.predict(X_test)
final_predictions = target_encoder.inverse_transform(test_pred_indices)

# Create DataFrame of test predictions and remove any duplicate IDs
test_preds_df = pd.DataFrame({
    ID_COL: test_ids,
    TARGET_COL: final_predictions
})
test_preds_df = test_preds_df.drop_duplicates(subset=[ID_COL], keep='first')

# Load exact sample submission file
sub = pd.read_csv(os.path.join(DATA_PATH, 'sample_submission.csv'))

if ID_COL in sub.columns:
    # Left merge on unique spectrum_id to guarantee exact length match
    sub = sub[[ID_COL]].merge(test_preds_df, on=ID_COL, how='left')
    sub[TARGET_COL] = sub[TARGET_COL].fillna(final_predictions[0])
else:
    # Slice exactly to required length
    sub[TARGET_COL] = final_predictions[:len(sub)]

sub.to_csv('submission.csv', index=False)
print("SUCCESS: submission.csv created!")

Training model...
Generating predictions...
SUCCESS: submission.csv created!


In [3]:
import os

if os.path.exists('submission.csv'):
    print(f"SUCCESS: submission.csv is ready. File size: {os.path.getsize('submission.csv')} bytes.")
else:
    print("ERROR: File was not generated.")

SUCCESS: submission.csv is ready. File size: 60017 bytes.
